In [ ]:
#import libraries
import numpy as np
import pandas as pd
#load data 
df = pd.read_csv('Orders_with_issues.csv')

#preview 
#print(" Sample Data:")
print(df.head())

#Summary
#print("\n Column Info:")
print(df.info())

#Check missing values
print("\n Missing values:")
print(df.isna().sum)

#Check unique shipping companies
#print("\n Unique Shipping Companies:")
print(df['ShippingCompany'].value_counts())


   OrderID CustomerID  OrderDate ShippedDate  ShippingCost ShipCountry  \
0   1000.0       C001  5/17/2025   7/30/2025  -57.25454602     Germany   
1   1001.0       C002  1/26/2025   7/30/2025        320.61      Canada   
2   1002.0       C003   3/8/2025   7/30/2025        165.17      Canada   
3   1003.0       C004  3/24/2025   7/30/2025         12.55     Germany   
4   1004.0       C005  4/15/2025   7/30/2025        186.36      Canada   

    ShipCity                 ShippingCompany  
0    Hamburg  Kiwilytics Goods Shipping LLC.  
1   Montreal                   UPS Worldwide  
2  Vancouver                 FedEx Logistics  
3     Munich            Aramex International  
4  Vancouver                 FedEx Logistics  
<class 'pandas.DataFrame'>
RangeIndex: 245 entries, 0 to 244
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          239 non-null    float64
 1   CustomerID       240 non-null    str 

In [4]:
#convert dates (Error Types:#coerce #ignore #raise)
df['OrderDate'] = pd.to_datetime(df['OrderDate'], errors='coerce')
df['ShippedDate'] = pd.to_datetime(df['ShippedDate'], errors='coerce')

#clean shipping cost 
df['ShippingCost'] = pd.to_numeric(df['ShippingCost'], errors='coerce')
df.loc[df['ShippingCost'] < 0, 'ShippingCost'] = np.nan

df['ShippingCost'] = df['ShippingCost'].fillna(df['ShippingCost'].median())



In [5]:
# Handle nulls 
df['OrderID'] = df['OrderID'].ffill()
df['CustomerID'] = df['CustomerID'].fillna('Unknown')
df['ShipCity'] = df['ShipCity'].fillna('Unspecified')

# Standardize names
df['ShippingCompany'] = df['ShippingCompany'].str.strip().str.title()
df['ShipCity'] = df['ShipCity'].str.strip().str.title()
df['ShipCountry'] = df['ShipCountry'].str.strip().str.title()

# Fix specific cases
df.loc[df['ShippingCompany'].str.contains("Kiwilytics", na=False), 'ShippingCompany'] = "Kiwilytics Goods Shipping LLC"

In [33]:
# Days between orders and shipment
df['DeliveryDays'] = df['ShippedDate'] - df['OrderDate']

# Flags
def get_status(x):
    if pd.isna(x):
        return "Unknown"
    elif x > 15:
        return "Late"
    else:
        return "OnTime"

# optional: convert to integer days and apply status
df['DeliveryDays'] = df['DeliveryDays'].dt.days
df['DeliveryStatus'] = df['DeliveryDays'].apply(get_status)

# Domestic vs International

def check_domestic(country) :
    if country in domestic_countries :
        return "Yes"
    else:
        return "No"
domestic_countries = ['Germany']
df['IsDomestic'] = df['ShipCountry'].apply(check_domestic)
df.head()

,OrderID,CustomerID,OrderDate,ShippedDate,ShippingCost,ShipCountry,ShipCity,ShippingCompany,DeliveryDays,DeliveryStatus,IsDomestic
0,1000.0,C001,2025-05-17,2025-07-30,NaN,Germany,Hamburg,Kiwilytics Goods Shipping LLC,74.0,Late,Yes
1,1001.0,C002,2025-01-26,2025-07-30,320.61,Canada,Montreal,Ups Worldwide,185.0,Late,No
2,1002.0,C003,2025-03-08,2025-07-30,165.17,Canada,Vancouver,Fedex Logistics,144.0,Late,No
3,1003.0,C004,2025-03-24,2025-07-30,12.55,Germany,Munich,Aramex International,128.0,Late,Yes
4,1004.0,C005,2025-04-15,2025-07-30,186.36,Canada,Vancouver,Fedex Logistics,106.0,Late,No


In [34]:
# Grouping examples
avg_shipping_by_company = df.groupby("ShippingCompany")['ShippingCost'].mean()
print("Avg Shipping Cost by Company:")
print(avg_shipping_by_company)

Avg Shipping Cost by Company:
ShippingCompany
Aramex International             258.805111
Dhl Express                      233.160000
Fedex Logistics                  229.533269
Kiwilytics Goods Shipping LLC    245.112162
Ups Worldwide                    258.632791
Name: ShippingCost, dtype: float64


In [39]:
# export cleaned file
df.to_csv("cleaned_orders_final.csv", index=False)

#Final Summary
print("\n Final Dataset snapshot:")
print(df.head())

print("\n Delivery Status Breakdown:")
print(df['DeliveryStatus'].value_counts())

print("\n Orders by Country:")
print(df['ShipCountry'].value_counts())

print("\Orders by City:")
print(df['ShipCity'].value_counts())

print("\n Top 3 SHippin Companies:")
print(df['ShippingCompany'].value_counts().head(3))






 Final Dataset snapshot:
   OrderID CustomerID  OrderDate ShippedDate  ShippingCost ShipCountry  \
0   1000.0       C001 2025-05-17  2025-07-30        234.09     Germany   
1   1001.0       C002 2025-01-26  2025-07-30        320.61      Canada   
2   1002.0       C003 2025-03-08  2025-07-30        165.17      Canada   
3   1003.0       C004 2025-03-24  2025-07-30         12.55     Germany   
4   1004.0       C005 2025-04-15  2025-07-30        186.36      Canada   

    ShipCity                ShippingCompany  DeliveryDays DeliveryStatus  \
0    Hamburg  Kiwilytics Goods Shipping LLC          74.0           Late   
1   Montreal                  Ups Worldwide         185.0           Late   
2  Vancouver                Fedex Logistics         144.0           Late   
3     Munich           Aramex International         128.0           Late   
4  Vancouver                Fedex Logistics         106.0           Late   

  IsDomestic  
0        Yes  
1         No  
2         No  
3        Yes

<>:14: SyntaxWarning: invalid escape sequence '\O'
<>:14: SyntaxWarning: invalid escape sequence '\O'
C:\Users\alfateh\AppData\Local\Temp\ipykernel_14268\246741474.py:14: SyntaxWarning: invalid escape sequence '\O'
  print("\Orders by City:")


ShippingCompany
Fedex Logistics         54
Dhl Express             53
Aramex International    51
Name: count, dtype: int64